<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Practica/AgenticAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Чем VS Code отличается от Colab (важно понять сразу)

В Colab вы пишете код в **ячейках** и нажимаете «Play» рядом с ячейкой. В VS Code всё иначе:

- **Код хранится в файлах** (например, `chat.py`), а не в ячейках.
- **Запуск происходит через терминал** — текстовое окно внизу редактора, куда вы вводите команды (как в командной строке Windows).
- **Проект — это папка.** Вы открываете в VS Code целую папку, и все файлы внутри неё становятся частью проекта.

Если это понять, дальше всё будет логично.

---

## Шаг 1. Установите Python (если его ещё нет)

Colab запускает Python «в облаке», а VS Code запускает его **на вашем компьютере**. Поэтому Python нужно установить локально.

1. Откройте [python.org/downloads](https://www.python.org/downloads/) и скачайте последнюю стабильную версию для Windows.
2. Запустите установщик. **Обязательно поставьте галочку «Add Python to PATH»** внизу первого экрана — это критически важно.
3. Нажмите «Install Now» и дождитесь завершения.
4. Проверьте установку. Откройте **PowerShell** (нажмите `Win`, начните вводить `PowerShell` и откройте его) и введите:
   ```powershell
   python --version
   ```
   Если вы увидели номер версии (например, `Python 3.13.x`) — всё хорошо. Если нет — переустановите Python, убедившись, что галочка «Add to PATH» стоит.

> **Альтернатива:** если у вас Windows 10/11, Python можно установить одной командой в PowerShell: `winget install Python.Python.3.14`. После установки закройте и снова откройте PowerShell.

---

## Шаг 2. Установите VS Code

1. Откройте [code.visualstudio.com](https://code.visualstudio.com/) и скачайте установщик для Windows.
2. Запустите его и установите как обычную программу (можно оставить все настройки по умолчанию).
3. После установки откройте VS Code.

---

## Шаг 3. Установите расширение Python в VS Code

Расширение добавляет подсветку синтаксиса, подсказки и кнопку запуска файлов.

1. В VS Code нажмите `Ctrl+Shift+X` (откроется панель Extensions).
2. В поиске введите `Python`.
3. Найдите расширение **Python** от Microsoft (у него значок с синей змейкой) и нажмите **Install**.

После установки VS Code сам найдёт ваш Python.

---

## Шаг 4. Установите Ollama (движок для моделей)

Ollama — это программа, которая запускает большие языковые модели локально и предоставляет к ним доступ через простой API.

1. Откройте **PowerShell** (можно прямо в VS Code, но проще в отдельном окне Windows).
2. Введите команду:
   ```powershell
   winget install Ollama.Ollama
   ```
   Это официальный способ установки для Windows через встроенный менеджер пакетов.
3. Дождитесь завершения. После установки Ollama **автоматически запустится в фоне** — в системном трее (рядом с часами) появится её значок.
4. **Закройте и снова откройте PowerShell** (или перезапустите VS Code), чтобы система «увидела» новую команду. Проверьте:
   ```powershell
   ollama --version
   ```
   Если вывелась версия — установка успешна.

---

## Шаг 5. Скачайте лёгкую модель

Модель — это «мозг», который будет отвечать на ваши вопросы. Начнём с **Phi-3 Mini** — она маленькая (около 2 ГБ), быстро работает и понимает русский.

В том же PowerShell выполните:
```powershell
ollama pull phi3:mini
```
Дождитесь окончания загрузки. Проверить, что модель появилась, можно командой:
```powershell
ollama list
```

---

## Шаг 6. Создайте папку для проекта

В VS Code проект — это папка на вашем компьютере.

1. Откройте «Проводник» Windows (значок папки на панели задач).
2. Перейдите, например, в `C:\Users\ВашеИмя\Documents`.
3. Создайте новую папку с именем `my_ai_chat` (без пробелов и русских букв — так надёжнее).

---

## Шаг 7. Откройте папку в VS Code

1. В VS Code в верхнем меню выберите **File → Open Folder…** (Файл → Открыть папку).
2. Найдите созданную папку `my_ai_chat` и нажмите «Выбрать папку».
3. Если VS Code спросит, доверяете ли вы авторам файлов, нажмите **Yes, I trust the authors**.

Теперь в левой панели (Explorer) отображается содержимое папки — пока пустое.

---

## Шаг 8. Создайте файл Python и напишите код

1. В левой панели наведите курсор на имя папки `MY_AI_CHAT` — появятся иконки.
2. Нажмите иконку **New File** (лист с плюсом).
3. Введите имя файла: `chat.py` и нажмите `Enter`.

Файл откроется в редакторе. Скопируйте туда следующий код:

```python
import requests

# Адрес чат-API Ollama на вашем компьютере
OLLAMA_URL = "http://localhost:11434/api/chat"

# Имя модели, которую мы скачали
MODEL = "phi3:mini"


def ask_model(messages):
    """Отправляет историю диалога модели и возвращает её ответ."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False,          # ждём ответ целиком
        "options": {"temperature": 0.7}
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    response.raise_for_status()
    # В ответе на /api/chat текст лежит в поле "message" -> "content"
    return response.json()["message"]["content"]


# История диалога: список словарей, по одному на каждую реплику
messages = [
    {"role": "system", "content": "Ты — дружелюбный помощник. Отвечай понятно и по существу."}
]

print("Чат с локальной моделью. Напишите вопрос или 'выход' для завершения.")

# Бесконечный цикл: программа ждёт ввода, пока вы не напишете "выход"
while True:
    user_text = input("\nВы: ")

    if user_text.strip().lower() in ("выход", "exit", "quit"):
        print("До свидания!")
        break

    if user_text.strip() == "":
        continue

    messages.append({"role": "user", "content": user_text})

    print("Модель думает... (первый ответ может занять до минуты)")
    try:
        answer = ask_model(messages)
    except requests.exceptions.ConnectionError:
        print("Ошибка: Ollama не запущена. Проверьте, что программа Ollama работает.")
        messages.pop()
        continue

    messages.append({"role": "assistant", "content": answer})
    print("\nМодель:", answer)
```

Этот код использует эндпоинт `/api/chat`, который хранит историю диалога, поэтому модель «помнит» контекст беседы.

---

## Шаг 9. Установите библиотеку `requests`

Код использует библиотеку `requests` для отправки HTTP-запросов. Её нужно установить один раз.

1. В VS Code откройте терминал: меню **Terminal → New Terminal** (или нажмите `` Ctrl+` ``).
2. Убедитесь, что в правом верхнем углу терминала выбрано **powershell**.
3. Выполните:
   ```powershell
   pip install requests
   ```

---

## Шаг 10. Запустите программу

Есть два равнозначных способа.

**Способ А (через терминал):**
В том же терминале введите:
```powershell
python chat.py
```

**Способ Б (кнопка Play):**
Откройте файл `chat.py` в редакторе и нажмите зелёный треугольник **▷** в правом верхнем углу — VS Code сам запустит файл в терминале.

После запуска в терминале появится:
```
Чат с локальной моделью. Напишите вопрос или 'выход' для завершения.

Вы: _
```
Курсор мигает после `Вы:` — программа ждёт ваш текст. Введите вопрос, нажмите `Enter`, и через некоторое время увидите ответ модели. Чтобы выйти, введите `выход`.

---

## Шаг 11. Если русские буквы превращаются в «кракозябры»

PowerShell иногда неправильно отображает кириллицу. Перед запуском программы выполните в том же терминале три команды:
```powershell
[Console]::OutputEncoding = [System.Text.Encoding]::UTF8
$OutputEncoding = [System.Text.Encoding]::UTF8
chcp 65001 > $null
```
После этого снова запустите `python chat.py`.
